In [2]:
import os
import json
import re

from openai import OpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [4]:
user_state = {

    "room_type": None,
    "area": None,
    "budget": None,

    "current_condition": {
        "repair_age": None,
        "wall_condition": None,
        "floor_condition": None,
        "ceiling_condition": None,
        "needs_replacement": [],
        "problems": []
    },

    "preferences": {
        "colors": [],
        "materials": [],
        "likes_details": None,
        "likes_minimalism": None,
        "likes_bright": None,
        "likes_natural": None,
        "favorite_examples": []
    },

    "style": {
        "selected": None,
        "confidence": 0,
        "alternatives": []
    },

    "design_plan": {

        "layout": {
            "proposal": None,
            "approved": False
        },

        "colors": {
            "proposal": None,
            "approved": False
        },

        "furniture": {
            "proposal": None,
            "approved": False
        },

        "lighting": {
            "proposal": None,
            "approved": False
        },

        "storage": {
            "proposal": None,
            "approved": False
        }
    },

    "stage": "discovery"
}

In [5]:
SYSTEM_PROMPT = """
Ты профессиональный AI-дизайнер интерьеров.

ТВОЯ РОЛЬ:
Ты не просто отвечаешь на вопросы.
Ты помогаешь человеку понять,
какой интерьер ему нужен.

ПРАВИЛА:

1. Общайся как дизайнер-консультант.

2. НЕ задавай сухие вопросы.

ПЛОХО:
"Какой стиль вам нравится?"

ХОРОШО:
"Вам ближе спокойные минималистичные
интерьеры или более декоративные
и выразительные?"

3. Если пользователь не знает стиль —
помоги определить его через предпочтения.

4. Постепенно согласовывай:
- планировку
- цвета
- мебель
- освещение
- хранение

5. НЕ переходи к следующему этапу,
пока пользователь не одобрит текущий.

6. Будь вовлекающим.
Помогай человеку визуализировать интерьер.

7. Всегда учитывай бюджет.

8. Узнавай текущее состояние помещения:
- как давно ремонт
- что нужно менять
- какие проблемы есть

9. Финальный текст должен быть
очень подробным и атмосферным.

10. Не пиши сухими списками.
Пиши как дизайнерскую концепцию.
"""

In [6]:
def ask_llm(prompt, temperature=0.7):

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

In [7]:
def clean_json(text):

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        return match.group(0)

    return text

In [10]:
def merge_dict(old, new):

    for k, v in new.items():

        if isinstance(v, dict) and k in old:
            merge_dict(old[k], v)

        else:
            if v not in [None, "", [], {}]:
                old[k] = v

In [8]:
def update_state(user_message, state):

    prompt = f"""
Извлеки информацию из сообщения пользователя.

ТЕКУЩИЙ STATE:
{json.dumps(state, ensure_ascii=False)}

СООБЩЕНИЕ:
{user_message}

Верни ТОЛЬКО JSON.

Не добавляй текст.

Обновляй только те поля,
которые пользователь упомянул.
"""

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Ты извлекаешь данные для state."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    content = clean_json(response.choices[0].message.content)

    try:

        new_state = json.loads(content)

        merge_dict(state, new_state)

        return state

    except Exception as e:

        print("JSON ERROR")
        print(content)

        return state

In [11]:
def detect_intent(message):

    prompt = f"""
Определи intent пользователя.

Варианты:

- answer
- approval
- rejection
- uncertainty
- ask_examples
- change_direction

Сообщение:
{message}

Верни только intent.
"""

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [12]:
STAGES = [
    "discovery",
    "style_detection",
    "layout",
    "colors",
    "furniture",
    "lighting",
    "storage",
    "final"
]

In [13]:
def discovery_stage(state):

    prompt = f"""
Ты знакомишься с человеком
и его помещением.

ТЕКУЩИЙ STATE:
{json.dumps(state, ensure_ascii=False)}

Задай ОДИН следующий важный вопрос.

Тебе нужно узнать:
- тип комнаты
- площадь
- бюджет
- текущее состояние
- проблемы помещения
- что нравится человеку

Общайся тепло и естественно.
"""

    return ask_llm(prompt)

In [14]:
def style_stage(state):

    prompt = f"""
Помоги пользователю понять,
какой стиль ему подходит.

ТЕКУЩИЙ STATE:
{json.dumps(state, ensure_ascii=False)}

Не спрашивай напрямую стиль.

Вместо этого:
- выясняй вкусы
- предлагай варианты
- сравнивай атмосферы

Задай ОДИН вопрос.
"""

    return ask_llm(prompt)

In [15]:
def generate_section_proposal(state, section):

    prompt = f"""
Создай предложение дизайнера
для раздела: {section}

STATE:
{json.dumps(state, ensure_ascii=False)}

Пиши:
- подробно
- атмосферно
- как дизайнер

Не списком советов.

Человек должен визуализировать интерьер.

В конце спроси:
нравится ли ему направление.
"""

    return ask_llm(prompt)

In [16]:
def generate_final_design(state):

    prompt = f"""
Создай полноценную концепцию ремонта.

STATE:
{json.dumps(state, ensure_ascii=False)}

Структура:

# Планировка
# Цветовая концепция
# Мебель
# Освещение
# Хранение
# Атмосфера интерьера
# Практические советы

Пиши очень подробно.

Не сухими пунктами.

Пользователь должен буквально
представить интерьер.
"""

    return ask_llm(prompt, temperature=0.8)

In [17]:
def next_stage(state):

    current = state["stage"]

    idx = STAGES.index(current)

    if idx < len(STAGES) - 1:
        state["stage"] = STAGES[idx + 1]

In [18]:
def chat(user_message):

    global user_state

    update_state(user_message, user_state)

    intent = detect_intent(user_message)

    stage = user_state["stage"]

    # DISCOVERY

    if stage == "discovery":

        required = [
            user_state["room_type"],
            user_state["area"],
            user_state["budget"]
        ]

        if all(required):
            next_stage(user_state)
            return style_stage(user_state)

        return discovery_stage(user_state)

    # STYLE DETECTION

    if stage == "style_detection":

        if user_state["style"]["selected"]:
            next_stage(user_state)

        return style_stage(user_state)

    # LAYOUT

    if stage == "layout":

        if not user_state["design_plan"]["layout"]["proposal"]:

            proposal = generate_section_proposal(
                user_state,
                "планировка"
            )

            user_state["design_plan"]["layout"]["proposal"] = proposal

            return proposal

        if intent == "approval":

            user_state["design_plan"]["layout"]["approved"] = True

            next_stage(user_state)

            return generate_section_proposal(
                user_state,
                "цветовая концепция"
            )

        return "Что хотелось бы изменить в планировке?"

    # COLORS

    if stage == "colors":

        if not user_state["design_plan"]["colors"]["proposal"]:

            proposal = generate_section_proposal(
                user_state,
                "цветовая концепция"
            )

            user_state["design_plan"]["colors"]["proposal"] = proposal

            return proposal

        if intent == "approval":

            user_state["design_plan"]["colors"]["approved"] = True

            next_stage(user_state)

            return generate_section_proposal(
                user_state,
                "мебель"
            )

        return "Что хотелось бы изменить в цветах?"

    # FURNITURE

    if stage == "furniture":

        if not user_state["design_plan"]["furniture"]["proposal"]:

            proposal = generate_section_proposal(
                user_state,
                "мебель"
            )

            user_state["design_plan"]["furniture"]["proposal"] = proposal

            return proposal

        if intent == "approval":

            user_state["design_plan"]["furniture"]["approved"] = True

            next_stage(user_state)

            return generate_section_proposal(
                user_state,
                "освещение"
            )

        return "Что хотелось бы изменить в мебели?"

    # LIGHTING

    if stage == "lighting":

        if not user_state["design_plan"]["lighting"]["proposal"]:

            proposal = generate_section_proposal(
                user_state,
                "освещение"
            )

            user_state["design_plan"]["lighting"]["proposal"] = proposal

            return proposal

        if intent == "approval":

            user_state["design_plan"]["lighting"]["approved"] = True

            next_stage(user_state)

            return generate_section_proposal(
                user_state,
                "хранение"
            )

        return "Что хотелось бы изменить в освещении?"

    # STORAGE

    if stage == "storage":

        if not user_state["design_plan"]["storage"]["proposal"]:

            proposal = generate_section_proposal(
                user_state,
                "хранение"
            )

            user_state["design_plan"]["storage"]["proposal"] = proposal

            return proposal

        if intent == "approval":

            user_state["design_plan"]["storage"]["approved"] = True

            next_stage(user_state)

            return generate_final_design(user_state)

        return "Что хотелось бы изменить в хранении?"

    # FINAL

    return generate_final_design(user_state)

In [19]:
while True:

    msg = input("Вы: ")

    if msg.lower() in ["exit", "quit"]:
        break

    response = chat(msg)

    print("\nAI дизайнер:\n")
    print(response)
    print("\n")


AI дизайнер:

Давайте начнем наше путешествие в мир дизайна! Чтобы лучше понять ваши нужды и создать идеальное пространство, мне бы очень хотелось узнать, какую именно комнату вы хотите преобразить? Например, это будет гостиная, спальня или, возможно, кухня? Также было бы здорово узнать о ее площади и бюджете, который вы готовы выделить на этот проект. Это поможет мне предложить вам наиболее подходящие решения!



AI дизайнер:

Давайте представим вашу кухню как сердце вашего дома, где собираются близкие и создаются уютные моменты. Чтобы лучше понять, как ее преобразить, расскажите, пожалуйста, о текущем состоянии вашей кухни. Как давно вы делали ремонт? Есть ли что-то, что вам не нравится или требует замены? Может быть, вы заметили какие-то проблемы, которые мешают вам наслаждаться этим пространством?



AI дизайнер:

Давайте начнем с вашего пространства! Как вы себя чувствуете на своей кухне? Есть ли что-то, что вас там совсем не устраивает или, наоборот, что-то, что вам нравится? Мо